In [ ]:
from openai import OpenAI
# macstudio
# client = OpenAI(api_key="0", base_url="http://192.168.0.103:1234/v1")
# 电信A100
client = OpenAI(api_key="0", base_url="http://192.168.106.55:20000/v1")

def llm_qwen(
    prompt, 
    model="Qwen3.5-27B",
    max_tokens=8192,
    temperature=0.7,
    top_p=0.95
):
    chat_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
    )
    return chat_response.choices[0].message.content

if __name__ == "__main__":
    llm_output = llm_qwen("你知道什么是Tetlock吗？")
    print(llm_output)

In [1]:
# -*- coding: utf-8 -*-
"""
Superforecast MVP - JSON 本地存储版 / Jupyter 单 cell 版
版本：v0.3-modular-decomposition

核心改动：
1. 第一层不再直接 build_contract。
2. 新增独立模块 question_decomposition_agents.py：多 Agent 识别云状问题，展开维度，生成多个 contract_candidates。
3. 对 cloud_judgment 自动走 multi_contract_portfolio。
4. 单个 contract 仍沿用原有 decomposition / evidence / panel / probability engine。
5. portfolio 使用 semantic_coverage_weight / measurability / intent_preservation / proxy_risk 做加权 logit 聚合。
6. 报告中明确展示：
   - question_type
   - semantic_collapse_risk
   - lost_dimensions
   - contract portfolio
   - 每个维度概率
   - 综合概率
"""

from __future__ import annotations

import copy
import json
import math
import re
import uuid
from datetime import date, datetime
from pathlib import Path
from statistics import mean, pstdev
from typing import Any, Dict, List, Tuple, Optional

from openai import OpenAI

import question_decomposition_agents as qd


# =============================================================================
# 0. 所有可调参数都在这里
# =============================================================================

CONFIG = {
    # -------------------------
    # LLM 服务配置
    # -------------------------
    "api_key": "0",
    # A100
    "base_url": "http://192.168.106.55:20000/v1",
    # macstudio
    # "base_url": "http://192.168.0.103:1234/v1",
    "model": "Qwen3.5-27B",

    # -------------------------
    # 本地 JSON 存储
    # -------------------------
    "storage_path": "./forecast_mvp_store.json",
    "question_decomposition_storage_path": "./question_decomposition_store.json",
    "save_question_decomposition_record": True,

    # -------------------------
    # LLM 通用参数
    # -------------------------
    "llm": {
        "max_tokens": 100000,
        "top_p": 0.95,
        "system": "You are a rigorous superforecasting research assistant. Output exactly what the user asks for.",
    },

    # -------------------------
    # 不同阶段的 temperature
    # -------------------------
    "temperature": {
        "question_expansion": 0.10,
        "contract": 0.10,
        "decomposition": 0.25,
        "evidence": 0.20,
        "panel": 0.55,
        "portfolio_summary": 0.20,
        "json_repair": 0.00,
    },

    # -------------------------
    # 云状问题识别配置
    # -------------------------
    "cloud_question": {
        "cloud_terms": [
            "更好", "更差", "改善", "恶化", "成功", "失败", "爆发", "崩溃",
            "变强", "变弱", "繁荣", "衰退", "领先", "落后", "有前途", "没前途",
            "过得好", "幸福", "压力", "景气", "复苏", "危机", "牛市", "熊市"
        ],
        "force_multi_contract_when_cloud": True,
        "min_intent_preservation_for_single_contract": 0.70,
        "high_risk_blocks_single_contract": True,
    },

    # -------------------------
    # Portfolio 聚合配置
    # -------------------------
    "portfolio": {
        "max_contract_candidates": 7,
        "min_contract_candidates": 3,

        # 如果模型没给 coverage，就按这些默认经济类维度权重近似
        "default_cloud_dimension_weights": {
            "output_or_gdp": 0.25,
            "income_and_employment": 0.20,
            "consumption_and_confidence": 0.15,
            "industry_and_business_vitality": 0.15,
            "housing_and_balance_sheet": 0.10,
            "fiscal_and_public_services": 0.05,
            "external_environment": 0.05,
            "subjective_wellbeing": 0.05,
        },

        # proxy risk 对组合权重的惩罚
        "proxy_risk_weight_factor": {
            "low": 1.00,
            "medium": 0.85,
            "high": 0.60,
            "unknown": 0.75,
        },

        # 组合置信区间
        "portfolio_interval": {
            "base_sigma": 0.35,
            "semantic_risk_penalty": {
                "low": 0.05,
                "medium": 0.25,
                "high": 0.55,
                "unknown": 0.35,
            },
            "subforecast_disagreement_weight": 0.45,
            "min_sigma": 0.35,
            "max_sigma": 2.20,
        },
    },

    # -------------------------
    # 预测引擎参数
    # -------------------------
    "engine": {
        "min_probability": 0.01,
        "max_probability": 0.99,

        "min_base_probability": 0.02,
        "max_base_probability": 0.98,

        "min_evidence_impact_log_odds": -0.80,
        "max_evidence_impact_log_odds": 0.80,

        "min_factor_effect_log_odds": -1.50,
        "max_factor_effect_log_odds": 1.50,

        "max_factors": 12,
        "max_evidence_cards": 12,
        "max_panel_agents": 8,

        "independence_penalty_power": 0.5,

        "ensemble_weights": {
            "base_const": 0.20,
            "base_low_forecastability_bonus": 0.25,
            "evidence_const": 0.10,
            "evidence_forecastability_bonus": 0.30,
            "causal_const": 0.10,
            "causal_forecastability_bonus": 0.25,
            "panel_const": 0.20,
        },

        "cold_start_min_history": 10,
        "cold_start_temperature_base": 1.25,
        "cold_start_temperature_forecastability_discount": 0.15,

        "brier_temperature": {
            "bad_threshold": 0.25,
            "weak_threshold": 0.22,
            "strong_threshold": 0.17,
            "bad_temperature": 1.45,
            "weak_temperature": 1.25,
            "normal_temperature": 1.12,
            "strong_temperature": 1.00,
        },

        "interval": {
            "base_sigma": 0.55,
            "forecastability_penalty": 1.15,
            "panel_disagreement_weight": 0.30,
            "evidence_quality_penalty": 0.65,
            "evidence_count_bonus_per_card": 0.03,
            "max_evidence_count_bonus": 0.25,
            "min_sigma": 0.35,
            "max_sigma": 2.00,
        },
    },

    # -------------------------
    # 输出控制
    # -------------------------
    "runtime": {
        "print_steps": True,
        "print_expansion_json": True,
        "print_intermediate_json": True,
        "print_subcontract_reports": False,
        "print_markdown": True,
        "print_raw_json": False,
    },
}


# 你在 ipynb 里主要改这里
# 中国在近20年是否会发生战争
# 2028年固态电池是否会普及？
QUESTION = """
未来三年中国普通人的生活会不会变好？
""".strip()

USER_EVIDENCE = """
""".strip()


# =============================================================================
# 1. LLM Client
# =============================================================================

client = OpenAI(
    api_key=CONFIG["api_key"],
    base_url=CONFIG["base_url"],
)


def llm_qwen(
    prompt: str,
    config: Dict[str, Any] = CONFIG,
    temperature: float = 0.30,
    max_tokens: Optional[int] = None,
    top_p: Optional[float] = None,
    system: Optional[str] = None,
) -> str:
    max_tokens = max_tokens or config["llm"]["max_tokens"]
    top_p = top_p if top_p is not None else config["llm"]["top_p"]
    system = system or config["llm"]["system"]

    chat_response = client.chat.completions.create(
        model=config["model"],
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return chat_response.choices[0].message.content or ""


# =============================================================================
# 2. 基础工具函数
# =============================================================================

def now_iso() -> str:
    return datetime.now().replace(microsecond=0).isoformat()


def short_id() -> str:
    return uuid.uuid4().hex[:12]


def clamp(x: Any, lo: float, hi: float) -> float:
    try:
        x = float(x)
    except Exception:
        x = 0.0
    return max(lo, min(hi, x))


def safe_float(x: Any, default: float = 0.0) -> float:
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def logit(p: float) -> float:
    p = clamp(p, 1e-5, 1 - 1e-5)
    return math.log(p / (1 - p))


def sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def probability_to_percent(p: float) -> str:
    return f"{float(p) * 100:.1f}%"


def json_dumps(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, indent=2)


def strip_think_blocks(text: str) -> str:
    return re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    ).strip()


def extract_json_object(text: str) -> Dict[str, Any]:
    text = strip_think_blocks(text).strip()

    fence = re.search(
        r"```(?:json)?\s*(.*?)\s*```",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    if fence:
        text = fence.group(1).strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
        return {"value": obj}
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        candidate = text[start:end + 1]
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict):
                return obj
            return {"value": obj}
        except Exception as e:
            raise ValueError(f"JSON 解析失败：{e}\n\n原始输出前2000字符：\n{text[:2000]}")

    raise ValueError(f"没有找到 JSON 对象。\n\n原始输出前2000字符：\n{text[:2000]}")


def llm_json(
    prompt: str,
    config: Dict[str, Any] = CONFIG,
    temperature: float = 0.20,
    max_tokens: Optional[int] = None,
) -> Dict[str, Any]:
    strict_prompt = prompt.strip() + """

硬性要求：
1. 只输出一个合法 JSON 对象。
2. 不要输出 Markdown。
3. 不要输出解释。
4. 不要输出代码块。
5. 所有字符串必须使用双引号。
"""
    raw = llm_qwen(
        strict_prompt,
        config=config,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    try:
        return extract_json_object(raw)
    except Exception:
        repair_prompt = f"""
下面是一段模型输出。请把它修复成一个合法 JSON 对象。

要求：
1. 只输出 JSON。
2. 不要解释。
3. 不要 Markdown。
4. 不要代码块。

原始输出：
{raw}
"""
        repaired = llm_qwen(
            repair_prompt,
            config=config,
            temperature=config["temperature"]["json_repair"],
            max_tokens=max_tokens,
        )
        return extract_json_object(repaired)


def normalize_weight_list(items: List[Dict[str, Any]], key: str) -> List[Dict[str, Any]]:
    vals = []
    for item in items:
        v = safe_float(item.get(key), 0.0)
        if v < 0:
            v = 0.0
        vals.append(v)

    s = sum(vals)
    if s <= 0:
        n = max(1, len(items))
        for item in items:
            item[key] = 1.0 / n
        return items

    for item, v in zip(items, vals):
        item[key] = v / s
    return items


def has_cloud_term(question: str, config: Dict[str, Any] = CONFIG) -> bool:
    q = str(question)
    for term in config["cloud_question"]["cloud_terms"]:
        if term in q:
            return True
    return False


def risk_factor(proxy_risk: str, config: Dict[str, Any] = CONFIG) -> float:
    m = config["portfolio"]["proxy_risk_weight_factor"]
    key = str(proxy_risk or "unknown").lower()
    return float(m.get(key, m["unknown"]))


def get_record_probability(record: Dict[str, Any]) -> float:
    if "portfolio_probability_engine" in record:
        return float(record["portfolio_probability_engine"]["calibrated_probability"])
    return float(record.get("probability_engine", {}).get("calibrated_probability", 0.5))


# =============================================================================
# 3. JSON 本地存储
# =============================================================================

def empty_store() -> Dict[str, Any]:
    return {
        "meta": {
            "name": "superforecast_mvp_json_store",
            "version": "0.2-cloud-expansion",
            "created_at": now_iso(),
            "updated_at": now_iso(),
        },
        "forecasts": [],
    }


def load_store(config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    path = Path(config["storage_path"])
    if not path.exists():
        return empty_store()

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            return empty_store()
        if "forecasts" not in data:
            data["forecasts"] = []
        if "meta" not in data:
            data["meta"] = {}
        return data
    except Exception:
        backup = path.with_suffix(path.suffix + f".broken.{datetime.now().strftime('%Y%m%d_%H%M%S')}")
        path.rename(backup)
        print(f"原 JSON 文件解析失败，已备份到：{backup}")
        return empty_store()


def save_store(store: Dict[str, Any], config: Dict[str, Any] = CONFIG) -> None:
    path = Path(config["storage_path"])
    path.parent.mkdir(parents=True, exist_ok=True)

    store.setdefault("meta", {})
    store["meta"]["updated_at"] = now_iso()
    store["meta"]["version"] = "0.2-cloud-expansion"

    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(store, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp_path.replace(path)


def save_forecast_record(record: Dict[str, Any], config: Dict[str, Any] = CONFIG) -> None:
    store = load_store(config)
    store.setdefault("forecasts", [])
    store["forecasts"].append(record)
    save_store(store, config)


def find_forecast(forecast_id: str, config: Dict[str, Any] = CONFIG) -> Optional[Dict[str, Any]]:
    store = load_store(config)
    for item in store.get("forecasts", []):
        if item.get("id") == forecast_id:
            return item
    return None


def update_forecast_record(forecast_id: str, patch: Dict[str, Any], config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    store = load_store(config)
    for idx, item in enumerate(store.get("forecasts", [])):
        if item.get("id") == forecast_id:
            item.update(patch)
            item["updated_at"] = now_iso()
            store["forecasts"][idx] = item
            save_store(store, config)
            return item
    raise ValueError(f"forecast_id 不存在：{forecast_id}")


def local_domain_brier_stats(domain: str, config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    store = load_store(config)
    scores = []
    for item in store.get("forecasts", []):
        item_domain = item.get("contract", {}).get("domain")
        if not item_domain:
            item_domain = item.get("portfolio_meta", {}).get("dominant_domain", "other")

        if (
            item.get("status") == "resolved"
            and item_domain == domain
            and item.get("brier_score") is not None
        ):
            scores.append(float(item["brier_score"]))

    if not scores:
        return {
            "n": 0,
            "avg_brier": None,
        }

    return {
        "n": len(scores),
        "avg_brier": mean(scores),
    }


# =============================================================================
# 4. Prompt Schema
# =============================================================================

# 第一阶段问题拆解 schema 已移到 question_decomposition_agents.py。
# 主流程只消费该模块输出的 final["contract_candidates"] / final["dimension_map"]。

DECOMPOSITION_SCHEMA = {
    "base_rate": {
        "reference_class": "最接近的参考类。",
        "base_probability": "0到1，作为先验概率。",
        "sample_size_guess": "如果没有硬数据，写 rough/unknown；不要假装精确。",
        "confidence": "0到1",
        "rationale": "为什么这个参考类合理。",
    },
    "factors": [
        {
            "factor": "影响因素名称",
            "sub_question": "可被单独判断的子问题",
            "direction_if_true": "increase | decrease | mixed",
            "probability": "该因素为真的概率，0到1",
            "effect_log_odds": "如果该因素为真，对主事件log-odds影响，-1.5到1.5",
            "confidence": "0到1",
            "dependencies": ["依赖的其他factor名称"],
            "rationale": "简短理由",
        }
    ],
    "known_unknowns": ["关键未知项"],
    "search_queries": ["如果未来接入搜索，应搜索的query"],
}

EVIDENCE_SCHEMA = {
    "evidence_cards": [
        {
            "claim": "证据主张。不能编造具体来源。",
            "source_type": "user_provided | background | official | news | paper | market | database | other",
            "source": "来源名称；如果没有真实来源，写 background_prior 或 user_input，不要写假URL。",
            "direction": "increase | decrease | neutral",
            "relevance": "0到1",
            "reliability": "0到1",
            "diagnosticity": "0到1，表示这条证据对预测是否有区分度",
            "freshness": "0到1",
            "independence_group": "同源证据用同一个group，避免重复计权",
            "impact_log_odds": "证据对主事件log-odds的裸影响，-0.8到0.8",
            "rationale": "为什么这么赋权",
        }
    ]
}

PANEL_SCHEMA = {
    "agent_forecasts": [
        {
            "agent_name": "outside_view | inside_view | skeptic | optimist | base_rate_guardian | domain_generalist",
            "probability": "0到1",
            "confidence": "0到1",
            "rationale": "简短理由",
            "strongest_update_trigger": "什么新证据会明显改变判断",
        }
    ]
}


# =============================================================================
# 5. Question Expansion / Contract 生成
# =============================================================================

def build_question_decomposition_config(config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    """
    把主预测流程 CONFIG 映射到独立拆题模块 CONFIG。
    这样主流程仍只维护一份 LLM 连接配置。
    """
    dec_config = copy.deepcopy(qd.CONFIG)

    dec_config["api_key"] = config.get("api_key", dec_config["api_key"])
    dec_config["base_url"] = config.get("base_url", dec_config["base_url"])
    dec_config["model"] = config.get("model", dec_config["model"])

    dec_config["llm"]["max_tokens"] = config.get("llm", {}).get(
        "max_tokens", dec_config["llm"]["max_tokens"]
    )
    dec_config["llm"]["top_p"] = config.get("llm", {}).get(
        "top_p", dec_config["llm"]["top_p"]
    )

    # 独立存储拆题记录；主预测记录仍写 forecast_mvp_store.json。
    dec_config["storage_path"] = config.get(
        "question_decomposition_storage_path",
        "./question_decomposition_store.json",
    )

    runtime = config.get("runtime", {})
    dec_config["runtime"]["print_steps"] = runtime.get("print_steps", True)
    dec_config["runtime"]["print_agent_outputs"] = runtime.get("print_expansion_json", True)
    dec_config["runtime"]["save_record"] = config.get("save_question_decomposition_record", True)

    return dec_config


def expand_question(question: str, config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    """
    主流程的第一步入口。

    v0.3 起，这里不再用单 Prompt 直接生成 normalized_question，
    而是调用独立的 Cloud-to-Contract 多 Agent 模块：
    question_decomposition_agents.run_question_decomposition_agents。

    返回值做了一层 legacy adapter，保持后面的 run_forecast / contract_from_candidate
    不需要大改。
    """
    dec_config = build_question_decomposition_config(config)
    decomposition_record = qd.run_question_decomposition_agents(question, dec_config)
    final = decomposition_record.get("final", {}) or {}
    audit = final.get("semantic_collapse_audit", {}) or {}

    # legacy adapter：后续主流程原本读取 top-level recommended_mode / allowed_single_contract。
    expansion = dict(final)
    expansion["semantic_collapse_risk"] = audit.get("semantic_collapse_risk", "unknown")
    expansion["collapse_reason"] = audit.get("collapse_reason", "")
    expansion["lost_dimensions_if_single_metric"] = audit.get("lost_dimensions_if_single_metric", [])
    expansion["recommended_mode"] = audit.get("recommended_mode", "single_contract")
    expansion["allowed_single_contract"] = audit.get("allowed_single_contract", True)
    expansion["recommended_user_choice_prompt"] = final.get("recommended_user_choice_prompt", "")

    # 只保留轻量级索引，完整 agent_outputs 由拆题模块单独落库。
    expansion["decomposition_record_id"] = decomposition_record.get("id")
    expansion["decomposition_module"] = decomposition_record.get("module")
    expansion["decomposition_validation_errors"] = decomposition_record.get("validation_errors", [])

    return expansion


def contract_from_candidate(candidate: Dict[str, Any], expansion: Dict[str, Any]) -> Dict[str, Any]:
    c = dict(candidate)

    contract = {
        "normalized_question": c.get("normalized_question", ""),
        "target_type": "binary",
        "deadline": c.get("deadline", ""),
        "resolution_criteria": c.get("resolution_criteria", ""),
        "resolution_sources": c.get("resolution_sources", []),
        "domain": c.get("domain", "other"),
        "forecastability_score": clamp(c.get("forecastability_score", 0.5), 0.0, 1.0),
        "ambiguity_flags": c.get("ambiguity_flags", []),
        "clarifying_assumptions": c.get("clarifying_assumptions", []),
        "reason": c.get("reason", ""),

        # 新增字段：用于防止语义坍缩
        "contract_id": c.get("contract_id", ""),
        "dimension_id": c.get("dimension_id", ""),
        "dimension_name": c.get("dimension_name", ""),
        "semantic_coverage_weight": clamp(c.get("semantic_coverage_weight", 0.0), 0.0, 1.0),
        "measurability": clamp(c.get("measurability", 0.5), 0.0, 1.0),
        "user_intent_preservation_score": clamp(c.get("user_intent_preservation_score", 0.5), 0.0, 1.0),
        "proxy_metric": c.get("proxy_metric", ""),
        "proxy_risk": c.get("proxy_risk", "unknown"),
        "lost_dimensions": c.get("lost_dimensions", []),

        "source_question_type": expansion.get("question_type", "mixed"),
        "source_semantic_collapse_risk": expansion.get("semantic_collapse_risk", "unknown"),
        "source_overall_semantic_intent": expansion.get("overall_semantic_intent", ""),
    }

    return contract


# =============================================================================
# 6. Agent 调用：分解、证据、Panel
# =============================================================================

def decompose_question(
    question: str,
    contract: Dict[str, Any],
    expansion: Optional[Dict[str, Any]] = None,
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    expansion_context = ""
    if expansion:
        expansion_context = f"""
原始问题的语义展开：
{json_dumps({
    "question_type": expansion.get("question_type"),
    "overall_semantic_intent": expansion.get("overall_semantic_intent"),
    "semantic_collapse_risk": expansion.get("semantic_collapse_risk"),
    "dimension_map": expansion.get("dimension_map"),
})}

注意：
当前只预测 portfolio 中的一个子合约。不要把该子合约误认为完整原问题。
"""

    prompt = f"""
你是 Superforecasting Decomposition Agent。使用 Tetlock 风格的方法拆解预测题。

方法：
1. outside view / base rate
2. inside view / 机制分析
3. Fermi decomposition
4. 反方视角
5. 关键未知项

原始问题：
{question}

{expansion_context}

当前预测合约：
{json_dumps(contract)}

要求：
1. base_probability 是先验，不是最终概率。
2. factors 的 effect_log_odds 是数学引擎后续会使用的数值；不要全部给极端值。
3. 明确 dependencies，避免假设所有子问题独立。
4. 如果没有硬数据，要在 rationale 里说明不确定性。
5. 不要编造具体统计数据。
6. search_queries 只是为以后接入搜索准备，不代表你已经搜索过。
7. 如果这是一个云状问题的子合约，必须只围绕当前 contract 的指标拆解，不要冒充完整原问题。

输出 JSON schema：
{json_dumps(DECOMPOSITION_SCHEMA)}
"""
    obj = llm_json(
        prompt,
        config=config,
        temperature=config["temperature"]["decomposition"],
    )

    engine = config["engine"]

    base = obj.get("base_rate", {}) or {}
    base["base_probability"] = clamp(
        base.get("base_probability", 0.5),
        engine["min_base_probability"],
        engine["max_base_probability"],
    )
    base["confidence"] = clamp(base.get("confidence", 0.5), 0.0, 1.0)
    base.setdefault("reference_class", "")
    base.setdefault("sample_size_guess", "unknown")
    base.setdefault("rationale", "")
    obj["base_rate"] = base

    clean_factors = []
    for f in (obj.get("factors") or [])[:engine["max_factors"]]:
        f["probability"] = clamp(
            f.get("probability", 0.5),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        f["effect_log_odds"] = clamp(
            f.get("effect_log_odds", 0.0),
            engine["min_factor_effect_log_odds"],
            engine["max_factor_effect_log_odds"],
        )
        f["confidence"] = clamp(f.get("confidence", 0.5), 0.0, 1.0)
        f.setdefault("factor", "")
        f.setdefault("sub_question", "")
        f.setdefault("direction_if_true", "mixed")
        f.setdefault("dependencies", [])
        f.setdefault("rationale", "")
        clean_factors.append(f)

    obj["factors"] = clean_factors
    obj.setdefault("known_unknowns", [])
    obj.setdefault("search_queries", [])

    return obj


def build_evidence_cards(
    question: str,
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    user_evidence: str = "",
    expansion: Optional[Dict[str, Any]] = None,
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    if user_evidence.strip():
        evidence_mode = "用户提供了证据材料，请优先从用户材料中抽取证据卡。"
    else:
        evidence_mode = "用户没有提供外部证据。只能生成 background 类型的分析性证据卡，不得编造新闻、URL、论文或具体数据。"

    expansion_context = ""
    if expansion:
        expansion_context = f"""
原始问题是经过语义展开后的预测组合。当前 evidence 只服务于当前子合约：
{json_dumps({
    "overall_semantic_intent": expansion.get("overall_semantic_intent"),
    "current_dimension": contract.get("dimension_name"),
    "proxy_metric": contract.get("proxy_metric"),
    "proxy_risk": contract.get("proxy_risk"),
})}
"""

    prompt = f"""
你是 Evidence Structuring Agent。你的任务不是写文章，而是把信息整理成可计算的 evidence_cards。

{evidence_mode}

原始问题：
{question}

{expansion_context}

预测合约：
{json_dumps(contract)}

问题拆解：
{json_dumps(decomposition)}

用户提供的证据材料：
{user_evidence if user_evidence.strip() else "<empty>"}

要求：
1. 证据卡数量 4 到 12 条。
2. 不要编造具体来源、URL、统计数据、新闻标题。
3. 如果没有真实来源，source_type 必须使用 background，source 写 background_prior。
4. 如果证据来自用户材料，source_type 写 user_provided，source 写 user_input。
5. impact_log_odds 是裸影响，后续程序会乘以 relevance/reliability/diagnosticity/freshness 等权重。
6. 多条同源或高度相关证据必须使用同一个 independence_group。
7. diagnosticity 代表证据的区分度；泛泛而谈的信息不要给高分。
8. 如果当前 contract 是单一代理指标，不要把证据说成覆盖完整原问题。

输出 JSON schema：
{json_dumps(EVIDENCE_SCHEMA)}
"""
    obj = llm_json(
        prompt,
        config=config,
        temperature=config["temperature"]["evidence"],
    )

    engine = config["engine"]

    clean_cards = []
    for c in (obj.get("evidence_cards") or [])[:engine["max_evidence_cards"]]:
        c["relevance"] = clamp(c.get("relevance", 0.5), 0.0, 1.0)
        c["reliability"] = clamp(c.get("reliability", 0.5), 0.0, 1.0)
        c["diagnosticity"] = clamp(c.get("diagnosticity", 0.5), 0.0, 1.0)
        c["freshness"] = clamp(c.get("freshness", 0.5), 0.0, 1.0)
        c["impact_log_odds"] = clamp(
            c.get("impact_log_odds", 0.0),
            engine["min_evidence_impact_log_odds"],
            engine["max_evidence_impact_log_odds"],
        )
        c.setdefault("claim", "")
        c.setdefault("source_type", "background")
        c.setdefault("source", "background_prior")
        c.setdefault("direction", "neutral")
        c.setdefault("independence_group", "default")
        c.setdefault("rationale", "")
        clean_cards.append(c)

    obj["evidence_cards"] = clean_cards
    return obj


def run_forecaster_panel(
    question: str,
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    evidence: Dict[str, Any],
    expansion: Optional[Dict[str, Any]] = None,
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    expansion_context = ""
    if expansion:
        expansion_context = f"""
这是原始云状问题的一个子合约，不是完整原问题。
原始语义：{expansion.get("overall_semantic_intent")}
当前维度：{contract.get("dimension_name")}
当前代理指标：{contract.get("proxy_metric")}
proxy_risk：{contract.get("proxy_risk")}
"""

    prompt = f"""
你是一个多智能体预测委员会。请让以下 6 个虚拟预测者独立给出概率：

1. outside_view：只重视参考类和历史基准。
2. inside_view：重视当前事件机制。
3. skeptic：专门寻找失败路径和反证。
4. optimist：专门寻找成功路径，但不能无脑乐观。
5. base_rate_guardian：防止过度偏离 base rate。
6. domain_generalist：综合判断。

重要：
这些概率不是最终答案。最终概率由 Python 数学引擎聚合。

原始问题：
{question}

{expansion_context}

预测合约：
{json_dumps(contract)}

拆解：
{json_dumps(decomposition)}

证据卡：
{json_dumps(evidence)}

要求：
1. 每个 agent 必须给 0 到 1 概率。
2. 不要互相抄同一个概率。
3. 给出 strongest_update_trigger。
4. 不要声称自己完成了实时搜索。
5. 如果这是云状问题的子合约，必须只预测当前子合约，不要把它说成完整原问题。

输出 JSON schema：
{json_dumps(PANEL_SCHEMA)}
"""
    obj = llm_json(
        prompt,
        config=config,
        temperature=config["temperature"]["panel"],
    )

    engine = config["engine"]

    clean_runs = []
    for r in (obj.get("agent_forecasts") or [])[:engine["max_panel_agents"]]:
        r["probability"] = clamp(
            r.get("probability", 0.5),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        r["confidence"] = clamp(r.get("confidence", 0.5), 0.0, 1.0)
        r.setdefault("agent_name", "agent")
        r.setdefault("rationale", "")
        r.setdefault("strongest_update_trigger", "")
        clean_runs.append(r)

    obj["agent_forecasts"] = clean_runs
    return obj


# =============================================================================
# 7. 概率引擎
# =============================================================================

def weight_evidence_cards(
    cards: List[Dict[str, Any]],
    config: Dict[str, Any] = CONFIG,
) -> Tuple[List[Dict[str, Any]], float, float]:
    engine = config["engine"]
    group_counts: Dict[str, int] = {}
    weighted_cards: List[Dict[str, Any]] = []
    total_delta = 0.0
    qualities = []

    penalty_power = engine["independence_penalty_power"]

    for c in cards:
        group = str(c.get("independence_group", "default"))
        group_counts[group] = group_counts.get(group, 0) + 1
        occurrence = group_counts[group]

        independence_penalty = 1.0 / (occurrence ** penalty_power)

        relevance = clamp(c.get("relevance", 0.5), 0.0, 1.0)
        reliability = clamp(c.get("reliability", 0.5), 0.0, 1.0)
        diagnosticity = clamp(c.get("diagnosticity", 0.5), 0.0, 1.0)
        freshness = clamp(c.get("freshness", 0.5), 0.0, 1.0)

        impact = clamp(
            c.get("impact_log_odds", 0.0),
            engine["min_evidence_impact_log_odds"],
            engine["max_evidence_impact_log_odds"],
        )

        quality_weight = relevance * reliability * diagnosticity * freshness * independence_penalty
        weighted_impact = impact * quality_weight

        c2 = dict(c)
        c2["independence_penalty"] = independence_penalty
        c2["quality_weight"] = quality_weight
        c2["weighted_impact_log_odds"] = weighted_impact

        weighted_cards.append(c2)
        total_delta += weighted_impact
        qualities.append(quality_weight)

    avg_quality = mean(qualities) if qualities else 0.0
    return weighted_cards, total_delta, avg_quality


def compute_causal_probability(
    base_p: float,
    factors: List[Dict[str, Any]],
    config: Dict[str, Any] = CONFIG,
) -> float:
    engine = config["engine"]
    x = logit(base_p)

    for f in factors[:engine["max_factors"]]:
        p_factor = clamp(
            f.get("probability", 0.5),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        effect = clamp(
            f.get("effect_log_odds", 0.0),
            engine["min_factor_effect_log_odds"],
            engine["max_factor_effect_log_odds"],
        )
        conf = clamp(f.get("confidence", 0.5), 0.0, 1.0)

        centered = (p_factor - 0.5) * 2.0
        x += centered * effect * conf

    return clamp(
        sigmoid(x),
        engine["min_probability"],
        engine["max_probability"],
    )


def compute_panel_probability(
    agent_runs: List[Dict[str, Any]],
    base_p: float,
    config: Dict[str, Any] = CONFIG,
) -> Tuple[float, float]:
    engine = config["engine"]

    ps = [
        clamp(
            r.get("probability", base_p),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        for r in agent_runs
    ]

    if not ps:
        return base_p, 0.0

    logits_all = [logit(p) for p in ps]
    logits_trimmed = sorted(logits_all)

    if len(logits_trimmed) >= 5:
        logits_trimmed = logits_trimmed[1:-1]

    panel_logit = mean(logits_trimmed)
    disagreement = pstdev(logits_all) if len(logits_all) > 1 else 0.0

    return (
        clamp(sigmoid(panel_logit), engine["min_probability"], engine["max_probability"]),
        disagreement,
    )


def ensemble_probabilities(
    base_p: float,
    evidence_p: float,
    causal_p: float,
    panel_p: float,
    forecastability: float,
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    engine = config["engine"]
    ew = engine["ensemble_weights"]
    f = clamp(forecastability, 0.0, 1.0)

    weights = {
        "base": ew["base_const"] + ew["base_low_forecastability_bonus"] * (1.0 - f),
        "evidence": ew["evidence_const"] + ew["evidence_forecastability_bonus"] * f,
        "causal": ew["causal_const"] + ew["causal_forecastability_bonus"] * f,
        "panel": ew["panel_const"],
    }

    total = sum(weights.values())
    weights = {k: v / total for k, v in weights.items()}

    x = (
        weights["base"] * logit(base_p)
        + weights["evidence"] * logit(evidence_p)
        + weights["causal"] * logit(causal_p)
        + weights["panel"] * logit(panel_p)
    )

    p = clamp(
        sigmoid(x),
        engine["min_probability"],
        engine["max_probability"],
    )

    return {
        "probability": p,
        "weights": weights,
    }


def calibrate_probability(
    raw_p: float,
    forecastability: float,
    local_stats: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Tuple[float, Dict[str, Any]]:
    engine = config["engine"]

    n = int(local_stats.get("n") or 0)
    avg_brier = local_stats.get("avg_brier")
    f = clamp(forecastability, 0.0, 1.0)

    if n < engine["cold_start_min_history"] or avg_brier is None:
        temperature = (
            engine["cold_start_temperature_base"]
            - engine["cold_start_temperature_forecastability_discount"] * f
        )
        mode = "cold_start_conservative_shrinkage"
    else:
        bt = engine["brier_temperature"]
        avg_brier = float(avg_brier)

        if avg_brier > bt["bad_threshold"]:
            temperature = bt["bad_temperature"]
        elif avg_brier > bt["weak_threshold"]:
            temperature = bt["weak_temperature"]
        elif avg_brier < bt["strong_threshold"]:
            temperature = bt["strong_temperature"]
        else:
            temperature = bt["normal_temperature"]

        mode = "local_brier_temperature"

    calibrated = sigmoid(logit(raw_p) / temperature)

    meta = {
        "calibration_mode": mode,
        "temperature": temperature,
        "domain_history_n": n,
        "domain_avg_brier": avg_brier,
    }

    return (
        clamp(calibrated, engine["min_probability"], engine["max_probability"]),
        meta,
    )


def confidence_interval(
    p: float,
    forecastability: float,
    panel_disagreement: float,
    avg_evidence_quality: float,
    evidence_count: int,
    config: Dict[str, Any] = CONFIG,
) -> Tuple[float, float, Dict[str, Any]]:
    engine = config["engine"]
    interval = engine["interval"]

    f = clamp(forecastability, 0.0, 1.0)
    q = clamp(avg_evidence_quality, 0.0, 1.0)

    n_bonus = min(
        interval["max_evidence_count_bonus"],
        interval["evidence_count_bonus_per_card"] * max(0, evidence_count - 4),
    )

    sigma = (
        interval["base_sigma"]
        + interval["forecastability_penalty"] * (1.0 - f)
        + interval["panel_disagreement_weight"] * min(panel_disagreement, 2.0)
        + interval["evidence_quality_penalty"] * (1.0 - q)
        - n_bonus
    )

    sigma = clamp(
        sigma,
        interval["min_sigma"],
        interval["max_sigma"],
    )

    lo = sigmoid(logit(p) - sigma)
    hi = sigmoid(logit(p) + sigma)

    return (
        clamp(lo, 0.0, 1.0),
        clamp(hi, 0.0, 1.0),
        {"logit_sigma": sigma},
    )


def compute_probability_bundle(
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    evidence_obj: Dict[str, Any],
    panel_obj: Dict[str, Any],
    local_stats: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    engine = config["engine"]

    base_p = clamp(
        (decomposition.get("base_rate") or {}).get("base_probability", 0.5),
        engine["min_base_probability"],
        engine["max_base_probability"],
    )

    forecastability = clamp(
        contract.get("forecastability_score", 0.5),
        0.0,
        1.0,
    )

    evidence_cards, total_evidence_delta, avg_quality = weight_evidence_cards(
        evidence_obj.get("evidence_cards") or [],
        config=config,
    )

    evidence_p = clamp(
        sigmoid(logit(base_p) + total_evidence_delta),
        engine["min_probability"],
        engine["max_probability"],
    )

    causal_p = compute_causal_probability(
        base_p,
        decomposition.get("factors") or [],
        config=config,
    )

    panel_p, disagreement = compute_panel_probability(
        panel_obj.get("agent_forecasts") or [],
        base_p,
        config=config,
    )

    ensemble = ensemble_probabilities(
        base_p,
        evidence_p,
        causal_p,
        panel_p,
        forecastability,
        config=config,
    )

    raw_p = ensemble["probability"]

    calibrated_p, calibration_meta = calibrate_probability(
        raw_p,
        forecastability,
        local_stats,
        config=config,
    )

    low, high, interval_meta = confidence_interval(
        calibrated_p,
        forecastability,
        disagreement,
        avg_quality,
        len(evidence_cards),
        config=config,
    )

    return {
        "base_probability": base_p,
        "evidence_probability": evidence_p,
        "causal_probability": causal_p,
        "panel_probability": panel_p,
        "raw_probability": raw_p,
        "calibrated_probability": calibrated_p,
        "confidence_low": low,
        "confidence_high": high,
        "weighted_evidence_cards": evidence_cards,
        "total_evidence_delta_log_odds": total_evidence_delta,
        "avg_evidence_quality": avg_quality,
        "panel_disagreement_logit_std": disagreement,
        "ensemble_weights": ensemble["weights"],
        "calibration": calibration_meta,
        "interval": interval_meta,
    }


# =============================================================================
# 8. 报告生成：单合约
# =============================================================================

def top_evidence(cards: List[Dict[str, Any]], k: int = 6) -> List[Dict[str, Any]]:
    return sorted(
        cards,
        key=lambda c: abs(safe_float(c.get("weighted_impact_log_odds"), 0.0)),
        reverse=True,
    )[:k]


def build_single_report_json(
    forecast_id: str,
    original_question: str,
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    evidence_obj: Dict[str, Any],
    panel_obj: Dict[str, Any],
    prob: Dict[str, Any],
    expansion: Optional[Dict[str, Any]] = None,
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    return {
        "id": forecast_id,
        "forecast_kind": "single_contract",
        "created_at": now_iso(),
        "updated_at": now_iso(),
        "status": "open",
        "outcome": None,
        "resolved_at": None,
        "brier_score": None,
        "original_question": original_question,
        "question_expansion": expansion,
        "contract": contract,
        "decomposition": decomposition,
        "evidence": {
            **evidence_obj,
            "evidence_cards": prob["weighted_evidence_cards"],
        },
        "panel": panel_obj,
        "probability_engine": {
            k: v
            for k, v in prob.items()
            if k not in {"weighted_evidence_cards"}
        },
        "runtime_config_snapshot": {
            "model": config["model"],
            "base_url": config["base_url"],
            "engine": config["engine"],
            "temperature": config["temperature"],
        },
    }


def build_single_markdown_report(report: Dict[str, Any]) -> str:
    contract = report["contract"]
    decomp = report["decomposition"]
    engine = report["probability_engine"]
    cards = report["evidence"]["evidence_cards"]
    panel = report.get("panel", {}).get("agent_forecasts", [])

    p = float(engine["calibrated_probability"])
    low = float(engine["confidence_low"])
    high = float(engine["confidence_high"])

    base_rate = decomp.get("base_rate", {}) or {}
    factors = decomp.get("factors", []) or []
    known_unknowns = decomp.get("known_unknowns", []) or []
    ambiguity_flags = contract.get("ambiguity_flags", []) or []
    assumptions = contract.get("clarifying_assumptions", []) or []

    lines = []

    lines.append(f"# Forecast {report['id']}")
    lines.append("")
    lines.append(f"**预测类型**：single_contract")
    lines.append(f"**预测题**：{contract.get('normalized_question', '')}")
    lines.append(f"**最终概率**：{probability_to_percent(p)}")
    lines.append(f"**粗略区间**：{probability_to_percent(low)} - {probability_to_percent(high)}")
    lines.append(f"**截止日期**：{contract.get('deadline', '')}")
    lines.append(f"**领域**：{contract.get('domain', 'other')}")
    lines.append(f"**可预测性评分**：{safe_float(contract.get('forecastability_score'), 0.5):.2f}")

    if contract.get("dimension_name"):
        lines.append("")
        lines.append("## 语义覆盖信息")
        lines.append(f"- 当前维度：{contract.get('dimension_name')}")
        lines.append(f"- 代理指标：{contract.get('proxy_metric')}")
        lines.append(f"- proxy_risk：{contract.get('proxy_risk')}")
        lines.append(f"- 原意保留分：{safe_float(contract.get('user_intent_preservation_score'), 0.0):.2f}")
        lines.append(f"- 语义覆盖权重：{safe_float(contract.get('semantic_coverage_weight'), 0.0):.2f}")
        lost = contract.get("lost_dimensions", []) or []
        if lost:
            lines.append(f"- 该子合约未覆盖：{', '.join(map(str, lost[:8]))}")

    lines.append("")
    lines.append("## 判定标准")
    lines.append(str(contract.get("resolution_criteria", "")))
    sources = contract.get("resolution_sources", []) or []
    if sources:
        lines.append(f"判定来源：{', '.join(map(str, sources))}")
    lines.append("")

    lines.append("## 概率计算拆解")
    lines.append(f"- Base rate prior：{probability_to_percent(engine['base_probability'])}")
    lines.append(f"- Evidence update 后：{probability_to_percent(engine['evidence_probability'])}")
    lines.append(f"- Causal factor 聚合后：{probability_to_percent(engine['causal_probability'])}")
    lines.append(f"- 多 Agent panel 聚合后：{probability_to_percent(engine['panel_probability'])}")
    lines.append(f"- Raw ensemble：{probability_to_percent(engine['raw_probability'])}")
    lines.append(f"- Calibrated final：{probability_to_percent(engine['calibrated_probability'])}")
    lines.append("")

    lines.append("## Ensemble 权重")
    weights = engine.get("ensemble_weights", {})
    for k, v in weights.items():
        lines.append(f"- {k}: {float(v):.3f}")
    lines.append("")

    lines.append("## 参考类 / Base Rate")
    lines.append(f"参考类：{base_rate.get('reference_class', '')}")
    lines.append(f"先验概率：{probability_to_percent(safe_float(base_rate.get('base_probability'), 0.5))}")
    lines.append(f"置信度：{safe_float(base_rate.get('confidence'), 0.5):.2f}")
    lines.append(f"说明：{base_rate.get('rationale', '')}")
    lines.append("")

    lines.append("## 关键因子")
    if factors:
        for f in factors[:8]:
            lines.append(
                f"- **{f.get('factor', '')}**："
                f"P={probability_to_percent(safe_float(f.get('probability'), 0.5))}，"
                f"effect_log_odds={safe_float(f.get('effect_log_odds'), 0.0):+.2f}，"
                f"方向={f.get('direction_if_true', '')}。"
                f"{f.get('rationale', '')}"
            )
    else:
        lines.append("- 无结构化因子。")
    lines.append("")

    lines.append("## 最有诊断价值的证据")
    top_cards = top_evidence(cards, k=6)
    if top_cards:
        for c in top_cards:
            lines.append(
                f"- [{c.get('direction', 'neutral')}] {c.get('claim', '')} "
                f"| weighted_log_odds={safe_float(c.get('weighted_impact_log_odds'), 0.0):+.3f} "
                f"| quality_weight={safe_float(c.get('quality_weight'), 0.0):.3f} "
                f"| source={c.get('source_type', '')}/{c.get('source', '')}"
            )
            rat = str(c.get("rationale", "")).strip()
            if rat:
                lines.append(f"  - {rat}")
    else:
        lines.append("- 无证据卡。")
    lines.append("")

    lines.append("## 多 Agent 分歧")
    if panel:
        for r in panel:
            lines.append(
                f"- **{r.get('agent_name', 'agent')}**："
                f"{probability_to_percent(safe_float(r.get('probability'), 0.5))}，"
                f"confidence={safe_float(r.get('confidence'), 0.5):.2f}。"
                f"{r.get('rationale', '')}"
            )
            trigger = str(r.get("strongest_update_trigger", "")).strip()
            if trigger:
                lines.append(f"  - 更新触发器：{trigger}")
    else:
        lines.append("- 无 panel 结果。")
    lines.append("")

    if known_unknowns:
        lines.append("## 关键未知项")
        for x in known_unknowns[:8]:
            lines.append(f"- {x}")
        lines.append("")

    if ambiguity_flags or assumptions:
        lines.append("## 模糊点与假设")
        for x in ambiguity_flags[:8]:
            lines.append(f"- 模糊点：{x}")
        for x in assumptions[:8]:
            lines.append(f"- 假设：{x}")
        lines.append("")

    cal = engine.get("calibration", {})
    lines.append("## 校准信息")
    lines.append(f"- calibration_mode：{cal.get('calibration_mode')}")
    lines.append(f"- temperature：{safe_float(cal.get('temperature'), 1.0):.3f}")
    lines.append(f"- domain_history_n：{cal.get('domain_history_n')}")
    lines.append(f"- domain_avg_brier：{cal.get('domain_avg_brier')}")
    lines.append("")

    lines.append("## 结论")
    lines.append(
        f"当前给出的概率是 **{probability_to_percent(p)}**。"
        f"这个数字由 base rate、证据 log-odds、因子聚合、多 Agent panel 和本地校准共同计算得到。"
    )

    return "\n".join(lines).strip() + "\n"


# =============================================================================
# 9. Portfolio 聚合
# =============================================================================

def compute_portfolio_probability(
    expansion: Dict[str, Any],
    sub_records: List[Dict[str, Any]],
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    items = []

    for r in sub_records:
        contract = r["contract"]
        engine = r["probability_engine"]

        p = float(engine["calibrated_probability"])
        lo = float(engine["confidence_low"])
        hi = float(engine["confidence_high"])

        semantic_w = safe_float(contract.get("semantic_coverage_weight"), 0.0)
        meas = safe_float(contract.get("measurability"), 0.5)
        intent = safe_float(contract.get("user_intent_preservation_score"), 0.5)
        proxy = risk_factor(contract.get("proxy_risk", "unknown"), config=config)

        raw_weight = semantic_w * (0.5 + 0.5 * meas) * (0.5 + 0.5 * intent) * proxy

        items.append({
            "forecast_id": r["id"],
            "contract_id": contract.get("contract_id", ""),
            "dimension_id": contract.get("dimension_id", ""),
            "dimension_name": contract.get("dimension_name", ""),
            "normalized_question": contract.get("normalized_question", ""),
            "proxy_metric": contract.get("proxy_metric", ""),
            "proxy_risk": contract.get("proxy_risk", "unknown"),
            "semantic_coverage_weight": semantic_w,
            "measurability": meas,
            "user_intent_preservation_score": intent,
            "risk_factor": proxy,
            "raw_portfolio_weight": raw_weight,
            "probability": p,
            "confidence_low": lo,
            "confidence_high": hi,
        })

    s = sum(max(0.0, x["raw_portfolio_weight"]) for x in items)
    if s <= 0:
        n = max(1, len(items))
        for x in items:
            x["portfolio_weight"] = 1.0 / n
    else:
        for x in items:
            x["portfolio_weight"] = max(0.0, x["raw_portfolio_weight"]) / s

    if not items:
        return {
            "calibrated_probability": 0.5,
            "confidence_low": 0.2,
            "confidence_high": 0.8,
            "portfolio_items": [],
            "method": "empty_fallback",
        }

    # logit 加权，而不是概率线性平均
    x = 0.0
    for item in items:
        x += item["portfolio_weight"] * logit(item["probability"])
    p = clamp(sigmoid(x), 0.01, 0.99)

    logits = [logit(item["probability"]) for item in items]
    disagreement = pstdev(logits) if len(logits) > 1 else 0.0

    interval_cfg = config["portfolio"]["portfolio_interval"]
    risk = str(expansion.get("semantic_collapse_risk", "unknown")).lower()
    risk_penalty = interval_cfg["semantic_risk_penalty"].get(
        risk,
        interval_cfg["semantic_risk_penalty"]["unknown"],
    )

    sigma = (
        interval_cfg["base_sigma"]
        + risk_penalty
        + interval_cfg["subforecast_disagreement_weight"] * min(disagreement, 2.0)
    )
    sigma = clamp(sigma, interval_cfg["min_sigma"], interval_cfg["max_sigma"])

    low = clamp(sigmoid(logit(p) - sigma), 0.0, 1.0)
    high = clamp(sigmoid(logit(p) + sigma), 0.0, 1.0)

    return {
        "calibrated_probability": p,
        "confidence_low": low,
        "confidence_high": high,
        "portfolio_items": items,
        "method": "weighted_logit_portfolio",
        "subforecast_logit_disagreement": disagreement,
        "portfolio_logit_sigma": sigma,
        "semantic_collapse_risk": expansion.get("semantic_collapse_risk", "unknown"),
    }


def build_portfolio_report_json(
    forecast_id: str,
    original_question: str,
    expansion: Dict[str, Any],
    sub_records: List[Dict[str, Any]],
    portfolio_engine: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    domains = []
    for r in sub_records:
        domains.append(r.get("contract", {}).get("domain", "other"))
    dominant_domain = max(set(domains), key=domains.count) if domains else "other"

    return {
        "id": forecast_id,
        "forecast_kind": "multi_contract_portfolio",
        "created_at": now_iso(),
        "updated_at": now_iso(),
        "status": "open",
        "outcome": None,
        "resolved_at": None,
        "brier_score": None,
        "original_question": original_question,
        "question_expansion": expansion,
        "sub_forecasts": sub_records,
        "portfolio_probability_engine": portfolio_engine,
        "portfolio_meta": {
            "dominant_domain": dominant_domain,
            "subforecast_count": len(sub_records),
        },
        "runtime_config_snapshot": {
            "model": config["model"],
            "base_url": config["base_url"],
            "portfolio": config["portfolio"],
            "engine": config["engine"],
            "temperature": config["temperature"],
        },
    }


def build_portfolio_markdown_report(report: Dict[str, Any]) -> str:
    expansion = report["question_expansion"]
    engine = report["portfolio_probability_engine"]
    items = engine.get("portfolio_items", [])
    sub_records = report.get("sub_forecasts", [])

    p = float(engine["calibrated_probability"])
    low = float(engine["confidence_low"])
    high = float(engine["confidence_high"])

    lines = []
    lines.append(f"# Forecast {report['id']}")
    lines.append("")
    lines.append("**预测类型**：multi_contract_portfolio")
    lines.append(f"**原始问题**：{report.get('original_question', '')}")
    lines.append(f"**综合概率**：{probability_to_percent(p)}")
    lines.append(f"**粗略区间**：{probability_to_percent(low)} - {probability_to_percent(high)}")
    lines.append(f"**问题类型**：{expansion.get('question_type')}")
    lines.append(f"**语义坍缩风险**：{expansion.get('semantic_collapse_risk')}")
    lines.append(f"**推荐模式**：{expansion.get('recommended_mode')}")
    lines.append("")

    lines.append("## 原始语义")
    lines.append(str(expansion.get("overall_semantic_intent", "")))
    collapse_reason = str(expansion.get("collapse_reason", "")).strip()
    if collapse_reason:
        lines.append("")
        lines.append(f"语义坍缩说明：{collapse_reason}")
    lost = expansion.get("lost_dimensions_if_single_metric", []) or []
    if lost:
        lines.append("")
        lines.append("如果强行单指标化，会丢失这些维度：")
        for x in lost[:10]:
            lines.append(f"- {x}")
    lines.append("")

    lines.append("## 维度展开")
    dims = expansion.get("dimension_map", []) or []
    if dims:
        for d in dims:
            lines.append(
                f"- **{d.get('dimension_name', d.get('dimension_id'))}** "
                f"| weight={safe_float(d.get('semantic_coverage_weight'), 0.0):.2f} "
                f"| measurability={safe_float(d.get('measurability'), 0.0):.2f} "
                f"| {d.get('meaning', '')}"
            )
    else:
        lines.append("- 无维度展开。")
    lines.append("")

    lines.append("## Contract Portfolio 结果")
    if items:
        for item in sorted(items, key=lambda x: x.get("portfolio_weight", 0.0), reverse=True):
            lines.append(
                f"- **{item.get('dimension_name')}** "
                f"| P={probability_to_percent(item.get('probability'))} "
                f"| weight={safe_float(item.get('portfolio_weight'), 0.0):.3f} "
                f"| proxy={item.get('proxy_metric')} "
                f"| proxy_risk={item.get('proxy_risk')}"
            )
            lines.append(f"  - 预测题：{item.get('normalized_question')}")
    else:
        lines.append("- 无 portfolio item。")
    lines.append("")

    lines.append("## 子预测摘要")
    for r in sub_records:
        c = r["contract"]
        e = r["probability_engine"]
        lines.append(
            f"- **{c.get('dimension_name', c.get('contract_id'))}**："
            f"{probability_to_percent(e.get('calibrated_probability'))} "
            f"区间 {probability_to_percent(e.get('confidence_low'))} - {probability_to_percent(e.get('confidence_high'))}"
        )
        lines.append(f"  - contract：{c.get('normalized_question')}")
        lines.append(
            f"  - base={probability_to_percent(e.get('base_probability'))}, "
            f"evidence={probability_to_percent(e.get('evidence_probability'))}, "
            f"causal={probability_to_percent(e.get('causal_probability'))}, "
            f"panel={probability_to_percent(e.get('panel_probability'))}"
        )
    lines.append("")

    lines.append("## 聚合方法")
    lines.append("- 使用 weighted logit portfolio，不是简单平均概率。")
    lines.append("- 权重由 semantic_coverage_weight、measurability、user_intent_preservation_score、proxy_risk 共同决定。")
    lines.append(f"- 子预测 logit 分歧：{safe_float(engine.get('subforecast_logit_disagreement'), 0.0):.3f}")
    lines.append(f"- portfolio logit sigma：{safe_float(engine.get('portfolio_logit_sigma'), 0.0):.3f}")
    lines.append("")

    lines.append("## 结论")
    lines.append(
        f"对原始云状问题，当前综合概率是 **{probability_to_percent(p)}**。"
    )

    return "\n".join(lines).strip() + "\n"


# =============================================================================
# 10. 主流程
# =============================================================================

def run_single_contract_forecast(
    question: str,
    contract: Dict[str, Any],
    user_evidence: str = "",
    expansion: Optional[Dict[str, Any]] = None,
    config: Dict[str, Any] = CONFIG,
    save: bool = False,
    print_report: bool = False,
    print_prefix: str = "",
) -> Dict[str, Any]:
    runtime = config["runtime"]

    def step(msg: str):
        if runtime.get("print_steps", True):
            print(f"{print_prefix}{msg}")

    step("[single 1/4] 拆解问题、生成参考类和因子...")
    decomposition = decompose_question(question, contract, expansion=expansion, config=config)
    if runtime.get("print_intermediate_json", True):
        print(f"{print_prefix}问题拆解:", json.dumps(decomposition, ensure_ascii=False, indent=2))

    step("[single 2/4] 结构化证据卡...")
    evidence_obj = build_evidence_cards(
        question,
        contract,
        decomposition,
        user_evidence=user_evidence,
        expansion=expansion,
        config=config,
    )
    if runtime.get("print_intermediate_json", True):
        print(f"{print_prefix}证据卡:", json.dumps(evidence_obj, ensure_ascii=False, indent=2))

    step("[single 3/4] 运行多 Agent 预测委员会...")
    panel_obj = run_forecaster_panel(
        question,
        contract,
        decomposition,
        evidence_obj,
        expansion=expansion,
        config=config,
    )
    if runtime.get("print_intermediate_json", True):
        print(f"{print_prefix}Panel 结果:", json.dumps(panel_obj, ensure_ascii=False, indent=2))

    step("[single 4/4] 概率聚合、校准...")
    domain = str(contract.get("domain", "other"))
    stats = local_domain_brier_stats(domain, config=config)

    prob = compute_probability_bundle(
        contract,
        decomposition,
        evidence_obj,
        panel_obj,
        stats,
        config=config,
    )

    forecast_id = short_id()

    report_json = build_single_report_json(
        forecast_id,
        question,
        contract,
        decomposition,
        evidence_obj,
        panel_obj,
        prob,
        expansion=expansion,
        config=config,
    )

    markdown_report = build_single_markdown_report(report_json)

    record = {
        **report_json,
        "markdown_report": markdown_report,
    }

    if save:
        save_forecast_record(record, config=config)

    if print_report:
        print("\n" + markdown_report)

    return record


def run_forecast(
    question: str,
    user_evidence: str = "",
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    runtime = config["runtime"]

    def step(msg: str):
        if runtime.get("print_steps", True):
            print(msg)

    step("[0/6] 原始问题语义展开...")
    expansion = expand_question(question, config=config)

    if runtime.get("print_expansion_json", True):
        print("语义展开:", json.dumps(expansion, ensure_ascii=False, indent=2))

    mode = expansion.get("recommended_mode", "single_contract")
    candidates = expansion.get("contract_candidates", []) or []

    # single contract
    if mode == "single_contract" and expansion.get("allowed_single_contract", True):
        step("[1/6] 使用 single_contract 模式...")
        contract = contract_from_candidate(candidates[0], expansion)

        if runtime.get("print_intermediate_json", True):
            print("最终预测合约:", json.dumps(contract, ensure_ascii=False, indent=2))

        record = run_single_contract_forecast(
            question,
            contract,
            user_evidence=user_evidence,
            expansion=expansion,
            config=config,
            save=True,
            print_report=runtime.get("print_markdown", True),
        )
        return record

    # ask_user_to_choose 暂时不阻塞；MVP 自动转 portfolio
    if mode == "ask_user_to_choose":
        if len(candidates) >= 2:
            step("[1/6] recommended_mode=ask_user_to_choose，但 MVP 自动使用 portfolio 模式继续...")
            mode = "multi_contract_portfolio"
        else:
            step("[1/6] recommended_mode=ask_user_to_choose，但只有一个候选合约，使用 single_contract fallback...")
            contract = contract_from_candidate(candidates[0], expansion)
            record = run_single_contract_forecast(
                question,
                contract,
                user_evidence=user_evidence,
                expansion=expansion,
                config=config,
                save=True,
                print_report=runtime.get("print_markdown", True),
            )
            return record

    # portfolio
    if mode in {"multi_contract_portfolio", "distribution_forecast", "mixed"} or not expansion.get("allowed_single_contract", True):
        step("[1/6] 使用 multi_contract_portfolio 模式...")

        max_n = config["portfolio"]["max_contract_candidates"]
        sub_records = []

        for idx, cand in enumerate(candidates[:max_n], start=1):
            contract = contract_from_candidate(cand, expansion)
            print("")
            step(f"[2-5/6] 子合约 {idx}/{min(len(candidates), max_n)}：{contract.get('dimension_name')}")

            if runtime.get("print_intermediate_json", True):
                print("子预测合约:", json.dumps(contract, ensure_ascii=False, indent=2))

            sub = run_single_contract_forecast(
                question,
                contract,
                user_evidence=user_evidence,
                expansion=expansion,
                config=config,
                save=False,
                print_report=runtime.get("print_subcontract_reports", False),
                print_prefix=f"[{idx}] ",
            )
            sub_records.append(sub)

        step("[6/6] 聚合 portfolio 概率并本地落库...")
        portfolio_engine = compute_portfolio_probability(
            expansion,
            sub_records,
            config=config,
        )

        forecast_id = short_id()
        report_json = build_portfolio_report_json(
            forecast_id,
            question,
            expansion,
            sub_records,
            portfolio_engine,
            config=config,
        )
        markdown_report = build_portfolio_markdown_report(report_json)

        record = {
            **report_json,
            "markdown_report": markdown_report,
        }

        save_forecast_record(record, config=config)

        if runtime.get("print_markdown", True):
            print("\n" + markdown_report)

        if runtime.get("print_raw_json", False):
            print("\n--- RAW JSON ---")
            print(json.dumps(record, ensure_ascii=False, indent=2))

        return record

    # 未知模式兜底
    step(f"[fallback] 未知 recommended_mode={mode}，使用第一个合约。")
    contract = contract_from_candidate(candidates[0], expansion)
    record = run_single_contract_forecast(
        question,
        contract,
        user_evidence=user_evidence,
        expansion=expansion,
        config=config,
        save=True,
        print_report=runtime.get("print_markdown", True),
    )
    return record


# =============================================================================
# 11. 本地管理函数
# =============================================================================

def list_forecasts(config: Dict[str, Any] = CONFIG, limit: int = 20) -> List[Dict[str, Any]]:
    store = load_store(config)
    items = store.get("forecasts", [])
    items = sorted(items, key=lambda x: x.get("created_at", ""), reverse=True)[:limit]

    for item in items:
        p = get_record_probability(item)
        p_str = probability_to_percent(p)

        status = item.get("status", "open")
        brier = item.get("brier_score")
        brier_str = f", brier={float(brier):.4f}" if brier is not None else ""

        kind = item.get("forecast_kind", "unknown")
        if kind == "multi_contract_portfolio":
            question = item.get("original_question", "")
        else:
            question = item.get("contract", {}).get("normalized_question", item.get("original_question", ""))

        print(f'{item.get("id")} | {item.get("created_at")} | {kind} | {p_str} | {status}{brier_str} | {question}')

    return items


def show_forecast(
    forecast_id: str,
    config: Dict[str, Any] = CONFIG,
    raw_json: bool = False,
) -> Dict[str, Any]:
    item = find_forecast(forecast_id, config=config)
    if item is None:
        raise ValueError(f"forecast_id 不存在：{forecast_id}")

    print(item.get("markdown_report", ""))

    if raw_json:
        print("\n--- RAW JSON ---")
        print(json.dumps(item, ensure_ascii=False, indent=2))

    return item


def resolve_forecast(
    forecast_id: str,
    outcome: int,
    config: Dict[str, Any] = CONFIG,
    force: bool = False,
) -> Dict[str, Any]:
    if outcome not in (0, 1):
        raise ValueError("outcome 只能是 0 或 1")

    item = find_forecast(forecast_id, config=config)
    if item is None:
        raise ValueError(f"forecast_id 不存在：{forecast_id}")

    if item.get("status") == "resolved" and not force:
        raise ValueError("该预测已经结算。如果要覆盖，设置 force=True。")

    p = get_record_probability(item)
    brier = (p - outcome) ** 2

    patch = {
        "status": "resolved",
        "outcome": outcome,
        "resolved_at": now_iso(),
        "brier_score": brier,
    }

    updated = update_forecast_record(forecast_id, patch, config=config)

    print(f"Resolved {forecast_id}: outcome={outcome}, p={p:.4f}, brier={brier:.4f}")
    return updated


def export_forecast(
    forecast_id: str,
    out_path: str,
    config: Dict[str, Any] = CONFIG,
    fmt: str = "md",
) -> str:
    item = find_forecast(forecast_id, config=config)
    if item is None:
        raise ValueError(f"forecast_id 不存在：{forecast_id}")

    path = Path(out_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if fmt == "json":
        path.write_text(json.dumps(item, ensure_ascii=False, indent=2), encoding="utf-8")
    elif fmt == "md":
        path.write_text(item.get("markdown_report", ""), encoding="utf-8")
    else:
        raise ValueError("fmt 只能是 md 或 json")

    print(f"已导出：{path}")
    return str(path)


def stability_test(
    question: str,
    user_evidence: str = "",
    n: int = 3,
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    """
    同一个问题跑 n 次，看第一层是否稳定。
    注意：会真实调用 LLM n 次，并保存 n 条记录。
    """
    results = []
    for i in range(n):
        print(f"\n========== stability run {i+1}/{n} ==========")
        r = run_forecast(question, user_evidence=user_evidence, config=config)
        results.append({
            "id": r["id"],
            "kind": r.get("forecast_kind"),
            "p": get_record_probability(r),
            "question_type": (r.get("question_expansion") or {}).get("question_type"),
            "semantic_collapse_risk": (r.get("question_expansion") or {}).get("semantic_collapse_risk"),
            "recommended_mode": (r.get("question_expansion") or {}).get("recommended_mode"),
            "contracts": [
                c.get("normalized_question")
                for c in (r.get("question_expansion") or {}).get("contract_candidates", [])
            ],
        })

    ps = [x["p"] for x in results]
    out = {
        "runs": results,
        "mean": mean(ps) if ps else None,
        "min": min(ps) if ps else None,
        "max": max(ps) if ps else None,
        "spread": (max(ps) - min(ps)) if ps else None,
    }
    print("\n========== stability summary ==========")
    print(json.dumps(out, ensure_ascii=False, indent=2))
    return out


# =============================================================================
# 12. 直接运行示例
# =============================================================================

print("原始问题:", QUESTION)
record = run_forecast(QUESTION, USER_EVIDENCE, CONFIG)

[0/11] Agent 0：问题类型分类...

===== Agent 0 分类结果 =====
{
  "question_type": "cloud_judgment",
  "is_cloud_question": true,
  "cloud_terms": [
    "普及"
  ],
  "triage": {
    "forecastability_zone": "cloudlike",
    "reason": "The term '普及' (popularize/widespread adoption) is a multidimensional concept lacking a single, universally agreed-upon quantitative threshold. It involves market share, production volume, cost parity, and consumer adoption rates, which vary by region and application (e.g., EVs vs. consumer electronics).",
    "requires_decomposition": true,
    "single_contract_allowed_by_type": false
  },
  "reason": "The question asks about the 'popularization' of solid-state batteries by 2028. 'Popularization' is a vague, cloud-like term that requires decomposition into measurable proxies (e.g., global shipment volume, percentage of EV market share, or cost per kWh) to be forecastable. It is not a simple binary event with a clear, objective trigger."
}
[1/11] Agent 1：解释原始意图...

===